# Technology — APM & Observability
---

<div style="padding:15px;border-left:8px solid #f093fb;background:#fdf0ff;border-radius:4px;">
<strong>Core Insight:</strong> Observability = three pillars: metrics (WHAT happened),
logs (WHY it happened), traces (WHERE in the stack it happened).
OpenTelemetry unifies all three with a vendor-neutral SDK. You instrument once — you export
to any backend: Prometheus, Datadog, Grafana, Jaeger.
</div>

### Why It Matters for Senior DE Roles
A Senior Data Engineer at Citi is expected to build and monitor production pipelines.
"Monitoring" is not optional — it's how you prove your pipeline is running correctly.
APM tools (AppDynamics, Dynatrace, CA APM) are tools you've used; knowing the conceptual
framework puts you ahead of candidates who only know metrics.

## 🧠 The Three Pillars of Observability

| Pillar | What it answers | Example tools | Example data |
|--------|----------------|---------------|-------------|
| **Metrics** | WHAT is happening (numbers) | Prometheus, Datadog, CloudWatch | CPU=72%, latency_p99=230ms, error_rate=0.1% |
| **Logs** | WHY it happened (events) | ELK Stack, Splunk, CloudWatch Logs | `ERROR 2024-01-15 14:23 job=etl msg="connection timeout after 30s"` |
| **Traces** | WHERE in the stack (flow) | Jaeger, Zipkin, AWS X-Ray | Request → API → DB → Cache → response: 230ms total, DB took 180ms |

### The Golden Signals (Google SRE)
Four signals that cover 90% of production monitoring needs:
1. **Latency** — how long requests take (p50, p95, p99 — not average)
2. **Traffic** — how many requests/events per second
3. **Errors** — rate of failed requests (500s, exceptions)
4. **Saturation** — how "full" your system is (CPU%, memory%, queue depth)

In [ ]:
# OpenTelemetry: instrument a Python pipeline
# pip install opentelemetry-api opentelemetry-sdk

from opentelemetry import trace, metrics
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.metrics import MeterProvider
import time, logging, json

# ── Setup (do once at startup) ───────────────────────────────────────────────
tracer_provider = TracerProvider()
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("capacity-pipeline")

meter_provider = MeterProvider()
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter("capacity-pipeline")

# Create metrics instruments
rows_processed = meter.create_counter(
    "pipeline.rows_processed",
    description="Total rows processed by the ETL pipeline"
)
pipeline_duration = meter.create_histogram(
    "pipeline.duration_seconds",
    description="Time to complete a pipeline run"
)
pipeline_errors = meter.create_counter(
    "pipeline.errors",
    description="Pipeline errors by type"
)

# ── Structured logging ───────────────────────────────────────────────────────
def log_event(level, msg, **kwargs):
    entry = {"timestamp": time.time(), "level": level, "msg": msg, **kwargs}
    print(json.dumps(entry))  # In prod: send to ELK/Splunk/CloudWatch

In [ ]:
# Instrumented ETL pipeline with traces + metrics + structured logs

def extract_monitoring_data(server_id: str, date: str) -> list:
    """Extract with distributed tracing."""
    with tracer.start_as_current_span("extract") as span:
        span.set_attribute("server.id", server_id)
        span.set_attribute("date", date)
        log_event("INFO", "Extracting data", server_id=server_id, date=date)
        # Simulate extraction
        time.sleep(0.1)
        data = [{"ts": i, "cpu": 45.0 + i*0.1} for i in range(1440)]  # 24h of minutely readings
        rows_processed.add(len(data), {"stage": "extract", "server": server_id})
        span.set_attribute("rows.extracted", len(data))
        return data

def transform_data(data: list) -> list:
    """Transform with error handling and metrics."""
    with tracer.start_as_current_span("transform") as span:
        start = time.perf_counter()
        try:
            transformed = [{"ts": row["ts"], "cpu_pct": round(row["cpu"], 2)} for row in data]
            rows_processed.add(len(transformed), {"stage": "transform"})
            return transformed
        except Exception as e:
            pipeline_errors.add(1, {"stage": "transform", "error_type": type(e).__name__})
            span.record_exception(e)
            log_event("ERROR", "Transform failed", error=str(e))
            raise

def run_pipeline(server_id: str, date: str):
    """Full pipeline with end-to-end trace."""
    with tracer.start_as_current_span("pipeline.run") as root_span:
        start = time.perf_counter()
        root_span.set_attribute("server.id", server_id)
        root_span.set_attribute("date", date)
        log_event("INFO", "Pipeline started", server=server_id, date=date)
        try:
            data = extract_monitoring_data(server_id, date)
            transformed = transform_data(data)
            duration = time.perf_counter() - start
            pipeline_duration.record(duration, {"server": server_id})
            log_event("INFO", "Pipeline complete", server=server_id, duration_s=round(duration, 3))
        except Exception as e:
            pipeline_errors.add(1, {"stage": "run", "server": server_id})
            log_event("ERROR", "Pipeline failed", server=server_id, error=str(e))
            raise

# Demo run
run_pipeline("SRV-1042", "2024-01-15")

## 📊 SLO / SLA / Error Budget

| Term | Definition | Example |
|------|------------|---------|
| **SLA** | Service Level Agreement — contractual guarantee | "99.9% uptime or we pay a penalty" |
| **SLO** | Service Level Objective — internal target | "p99 latency < 500ms for 99.9% of requests" |
| **SLI** | Service Level Indicator — the measurement | "Measured p99 = 312ms over last 30 days" |
| **Error Budget** | How much SLO you can miss | `error_budget = 1 - SLO = 0.1% = 43.8 min/month` |

### The Error Budget Formula
```
SLO = 99.9%  →  Error Budget = 0.1%  =  ~43.8 min/month downtime allowed
SLO = 99.99% →  Error Budget = 0.01% =  ~4.4 min/month
SLO = 99.5%  →  Error Budget = 0.5%  =  ~3.65 hr/month
```

**How to use it:** If you've consumed 80% of your monthly error budget in 2 weeks,
STOP all non-critical deployments. The error budget is your risk quota.

### Trend-Based vs Threshold Alerting
- **Static threshold:** Alert when CPU > 90%. Problem: server at 90% for 6 months = not an emergency.
- **Trend-based:** Alert when CPU jumps from 40% to 85% overnight (deviation from trend).
- Trend-based dramatically reduces false positives — the key insight from the Citi APM redesign.

In [ ]:
import numpy as np

# ══════════════════════════════════════
# TREND-BASED ALERTING — the Citi APM approach
# ══════════════════════════════════════

# 90-day baseline of CPU readings for one server
np.random.seed(42)
days = 90
baseline_cpu = np.random.normal(42, 5, size=days)  # stable ~42% CPU

# Simulate a sudden spike on day 85
current_week = np.concatenate([baseline_cpu[:84], np.array([78, 82, 85, 88, 90, 87, 89])])

# ── Static threshold (old approach) ─────────────────────────────────────────
STATIC_THRESHOLD = 85
static_alerts = current_week > STATIC_THRESHOLD
print(f"Static threshold alerts (> {STATIC_THRESHOLD}%): {static_alerts.sum()} days")

# ── Trend-based alerting (new approach) ─────────────────────────────────────
# Rolling 14-day baseline
window = 14
rolling_baseline = np.array([
    current_week[max(0, i-window):i].mean()
    for i in range(1, len(current_week)+1)
])

# Alert when current reading deviates > 20pp from rolling baseline
DEVIATION_THRESHOLD = 20
deviation = current_week - rolling_baseline
trend_alerts = deviation > DEVIATION_THRESHOLD
print(f"Trend-based alerts (deviation > {DEVIATION_THRESHOLD}pp): {trend_alerts.sum()} days")
print(f"Alert reduction: {static_alerts.sum()} → {trend_alerts.sum()}")

# The spike is caught (days 85-90 deviate sharply from baseline)
# Servers stable at high utilization do NOT alert (no deviation)
print(f"
Deviations on last 7 days: {deviation[-7:].round(1)}")
print("Days 85-90 flagged:", trend_alerts[-7:])

## 🎤 5 Interview Q&A

**Q1: What are the three pillars of observability and what does each answer?**
Metrics (WHAT): numbers over time — CPU%, latency_p99, error_rate.
Logs (WHY): structured events with context — what happened at what time with what parameters.
Traces (WHERE): how a request flowed through services — which service took the most time.
You need all three: metrics tell you there's a problem, logs tell you why, traces show where.

---

**Q2: What is an SLO and how do you calculate an error budget?**
SLO = Service Level Objective — an internal target for service reliability. Example: 99.9% availability.
Error budget = 1 - SLO = 0.1% = 43.8 minutes of downtime allowed per month.
Error budgets create a risk management framework: if 80% consumed in 2 weeks, stop risky deployments.
They align engineering (ship features) vs ops (maintain stability).

---

**Q3: What is OpenTelemetry and why is it important?**
OpenTelemetry is a CNCF vendor-neutral observability framework. You instrument your code once
with the OTel SDK, then export to any backend: Prometheus, Datadog, Jaeger, AWS X-Ray.
Avoids vendor lock-in. Your instrumentation code doesn't change when you switch from
Datadog to Prometheus. At Citi, migrating from CA APM to Dynatrace required zero code
changes to instrumented applications.

---

**Q4: What is the difference between p50, p95, and p99 latency?**
p50 = median — 50% of requests finish in this time or less. p95 = 95% of requests are ≤ this.
p99 = 99% of requests are ≤ this. Average hides outliers — a p99 of 2 seconds means 1% of users
wait > 2 seconds. In a system handling 1,000 requests/second, that's 10 users/second experiencing
a 2-second wait. Always monitor percentiles, not averages.

---

**Q5: What's the problem with static threshold alerting and how do you fix it?**
Static thresholds fire whenever a metric crosses a fixed line, regardless of normal behavior.
A server that normally runs at 90% CPU is not an emergency — but a static 85% alert fires constantly.
This causes alert fatigue: ops teams start ignoring alerts because false positive rate is too high.
Fix: trend-based alerting. Alert when the reading deviates significantly from its rolling baseline
(e.g., > 20 percentage points from 14-day rolling average). This catches real anomalies while
ignoring stable-but-high systems.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Observability** | The ability to understand a system's internal state from its external outputs (metrics, logs, traces) |
| **APM** | Application Performance Management — monitoring tools for application health (CA APM, AppDynamics, Dynatrace) |
| **SLO** | Service Level Objective — internal reliability target (e.g., 99.9% uptime) |
| **SLA** | Service Level Agreement — contractual guarantee to customers |
| **Error Budget** | `1 - SLO` — how much unreliability you're allowed; consumed by incidents and deployments |
| **SLI** | Service Level Indicator — the measured metric that determines if you're meeting your SLO |
| **OpenTelemetry (OTel)** | Vendor-neutral observability framework — one SDK, any backend |
| **Trace** | End-to-end record of a request's journey through services, showing timing per component |
| **Span** | A single unit of work within a trace — one service call or database query |
| **Golden Signals** | Google SRE's four key metrics: Latency, Traffic, Errors, Saturation |
| **Alert Fatigue** | When ops teams stop trusting alerts because false positive rate is too high |
| **Trend-Based Alerting** | Alert on deviation from rolling baseline rather than absolute threshold |

## 💼 The Citi Narrative

**Context:** Citi APM team monitoring 6,000+ endpoints across 50+ applications.
Tools: CA APM (Introscope) → AppDynamics → Dynatrace migration.

**The Problem:** Static threshold alerts — 400+ alerts/day firing. Ops team response rate was
near zero because > 70% were false positives. A server at 90% CPU for 6 months was alerting daily.
Real incidents were buried in noise.

**The Fix — Trend-Based Alerting:**
1. Pulled 90-day baselines from CA APM API into PostgreSQL
2. Computed rolling 14-day average per server per metric
3. Changed alert condition: `current_value - rolling_avg > threshold` instead of `current_value > threshold`
4. Added SQL window functions to compute the baseline continuously

**Result:** Alert volume dropped from 400+/day to ~120/day (70% reduction).
More importantly, real incidents were now caught earlier — a server deviating 30pp from
its baseline at 2am is a real problem, even if the absolute value is only 55%.

**Interview Line:** *"The insight was that the metric's absolute value is less informative
than its deviation from its own history. A server at 90% CPU that's been there for 6 months
is fine. A server that jumped from 40% to 85% overnight is on fire. We built the trending
model in Python, stored baselines in PostgreSQL, and used SQL window functions to compute
the rolling averages. Alert volume dropped 70% while catching more real incidents."*

In [ ]:
# Practice: Build a simple alert evaluation system

import numpy as np

def evaluate_alerts(current_readings: np.ndarray,
                    historical_baseline: np.ndarray,
                    static_threshold: float = 85.0,
                    deviation_threshold: float = 20.0) -> dict:
    """
    Compare static threshold vs trend-based alerting.

    Args:
        current_readings: Array of current metric values (one per server)
        historical_baseline: Array of 30-day average per server
        static_threshold: Alert if current > this value
        deviation_threshold: Alert if deviation from baseline > this value

    Returns:
        Dict with alert counts and server indices for each method
    """
    # Your implementation here:
    static_alerts = None       # servers where current > static_threshold
    trend_alerts = None        # servers where (current - baseline) > deviation_threshold

    # Solution:
    static_alerts = np.where(current_readings > static_threshold)[0]
    deviations = current_readings - historical_baseline
    trend_alerts = np.where(deviations > deviation_threshold)[0]

    return {
        "static_count": len(static_alerts),
        "trend_count": len(trend_alerts),
        "static_servers": static_alerts,
        "trend_servers": trend_alerts,
        "reduction_pct": round((1 - len(trend_alerts)/max(len(static_alerts),1)) * 100, 1)
    }

# Test it
np.random.seed(42)
n_servers = 6000
baseline = np.random.normal(42, 10, n_servers)  # historical averages
current = baseline + np.random.normal(0, 5, n_servers)  # small noise
# Inject 50 real incidents (big spike)
incident_idx = np.random.choice(n_servers, 50, replace=False)
current[incident_idx] += np.random.uniform(25, 45, 50)

result = evaluate_alerts(current, baseline)
print(f"Static threshold alerts:  {result['static_count']}")
print(f"Trend-based alerts:       {result['trend_count']}")
print(f"Alert reduction:          {result['reduction_pct']}%")
print(f"(Real incidents injected: 50)")

## 🎯 Summary

### The Pattern
**Observability** — Metrics + Logs + Traces = full system visibility.
**Trend-based alerting** = alert on deviation from baseline, not absolute values.

### The Stack
| Layer | Tool examples | What you learn |
|-------|--------------|----------------|
| Metrics | Prometheus, CloudWatch, Datadog | Dashboards, SLO tracking |
| Logs | ELK, Splunk, CloudWatch Logs | Root cause investigation |
| Traces | Jaeger, AWS X-Ray, Datadog APM | Latency bottleneck identification |
| Framework | OpenTelemetry | Vendor-neutral instrumentation |

### Interview Confidence Checklist
- [ ] Can name the three pillars and what each answers
- [ ] Can explain SLO / SLA / Error Budget and the formula
- [ ] Can explain trend-based vs static threshold alerting
- [ ] Can name the Citi narrative: 70% alert reduction, APM migration
- [ ] Can describe OpenTelemetry's value proposition

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀